# 00 - ECG-CXR bridge configuration

This clean pipeline uses Symile-MIMIC only to learn an ECG↔CXR reference bridge. It does not use laboratory measurements, ECG diagnoses, ECG reports, CXR reports, or CheXpert labels as training targets.

Run order: 00, 01, 02, 03, 04, 05, 06, 07. Notebook 08 documents and validates the future external MIMIC adapter inputs.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import platform
import sys

import torch

SYMILE_ROOT = Path(os.environ.get(
    "SYMILE_ROOT",
    "/path/to/symile-mimic/1.0.0",
))
ARTIFACT_ROOT = Path(os.environ.get(
    "ECG_CXR_BRIDGE_ARTIFACT_ROOT",
    "/path/to/ecg_cxr_bridge_artifacts",
))
HF_CACHE = Path(os.environ.get(
    "HF_HOME", "/path/to/huggingface_cache"
))

CONFIG = {
    "pipeline_version": "symile_ecg_cxr_bridge_v1",
    "symile_root": str(SYMILE_ROOT),
    "artifact_root": str(ARTIFACT_ROOT),
    "hf_cache": str(HF_CACHE),
    "hubert_model_id": "Edoardo-BS/hubert-ecg-base",
    "cxr_model_id": "microsoft/rad-dino",
    "splits": ["train", "val", "val_retrieval", "test"],
    "seed": 43,
}

for directory in [
    ARTIFACT_ROOT, ARTIFACT_ROOT / "manifests",
    ARTIFACT_ROOT / "features", ARTIFACT_ROOT / "models",
    ARTIFACT_ROOT / "code", ARTIFACT_ROOT / "reports",
    ARTIFACT_ROOT / "external", HF_CACHE,
]:
    directory.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
config_path = ARTIFACT_ROOT / "config.json"
config_path.write_text(json.dumps(CONFIG, indent=2))
print(json.dumps(CONFIG, indent=2))
print({"python": sys.version.split()[0], "platform": platform.platform(),
       "torch": torch.__version__, "cuda": torch.cuda.is_available()})
print("Saved:", config_path)


In [ ]:
required_packages = ["numpy", "pandas", "PIL", "torch", "tqdm", "transformers"]
missing = [name for name in required_packages if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError("Install missing packages in the Biowulf environment: " + ", ".join(missing))
assert SYMILE_ROOT.exists(), f"Missing Symile root: {SYMILE_ROOT}"
print("Environment and Symile root are ready.")
